# Hub-and-Spoke Multi-Agent Memory Evaluation

## The Problem

In a multi-agent system, the most important failures happen *before* the final answer is produced. A sub-agent may receive stale context, the coordinator may update the budget but not re-dispatch the hotel search, or two agents may work from different versions of the same plan. These are **memory and coordination failures** — and they're invisible unless you instrument for them.

## What This Notebook Does

We run a **travel planning** hub-and-spoke system through two sessions:

1. **Simple Session** — User requests a trip with a fixed budget. No changes mid-conversation. This is the baseline.
2. **Session with Feedback** — Same initial request, but the user changes the budget mid-session. This tests whether agents pick up the update, whether the coordinator re-dispatches correctly, and whether the final itinerary reflects the new constraints.

The metrics tell us what happened — no manual labeling needed.

## Architecture

```
                    ┌─────────────────────┐
                    │   User / Customer    │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Travel Coordinator │  ← Hub (no memory)
                    │   Decides which      │    Routes queries to spokes
                    │   spoke to call      │    Compresses user message
                    └──┬───────┬────────┬──┘    into handoff query
                       │       │        │
              ┌────────▼──┐ ┌──▼─────┐ ┌▼──────────┐
              │  Flight   │ │ Hotel  │ │ Itinerary │
              │  Agent    │ │ Agent  │ │ Agent     │
              └─────┬─────┘ └───┬────┘ └─────┬─────┘
                    │           │             │
              ┌─────▼───────────▼─────────────▼─────┐
              │     Shared Memory (Python list)     │
              │     Each agent reads prior entries   │
              │     and appends its own response     │
              └─────────────────────────────────────┘
```

**Flow:**
- Coordinator receives user request and delegates via tool calls
- Each spoke reads shared memory (a Python list), processes the query, appends its response
- Itinerary agent runs last — consumes Flight + Hotel outputs from memory
- All context flows through the coordinator or through shared memory

## Metrics Evaluated

**Memory Context Metrics** (LLM-as-judge, 1-5 scale):

| Metric | What it measures |
|--------|------------------|
| Context Freshness | Is the agent working with the latest information? |
| Handoff Completeness | Did the coordinator include all facts the spoke needs? |
| Context Utilization | Did the spoke use the context it read from memory? |
| State Consistency | Do all spokes agree on key facts (budget, dates, etc.)? |
| Memory Write Accuracy | Is what the agent wrote to memory factually correct? |
| Redundant Context | How much repeated/irrelevant context was transferred? |
| CCR | Length ratio of handoff vs original (pure math) |

**Memory Latency Metrics** (timers + token counts):

| Metric | What it measures |
|--------|------------------|
| Memory Read/Write Latency | Time for memory operations |
| Coordination Latency % | Fraction of time spent on coordination vs reasoning |
| Coordination Token % | Fraction of tokens spent on context vs generation |

## Step 1 — Setup

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import os
import time
import logging
from datetime import datetime
from IPython.display import display, Markdown

from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

from metrics_collector import MetricsCollector
from model_config import (
    AGENT_MODEL_ID, FLIGHT_PROMPT, HOTEL_PROMPT,
    ITINERARY_PROMPT, HUB_PROMPT,
)

region = os.getenv("AWS_REGION", "us-west-2")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("hub-spoke-metrics")

## Step 2 — Shared Memory (Python list)

No AgentCore Memory, no AWS setup, no 3-minute waits. Shared memory is a simple list that agents read from and append to. Each entry is tagged with the agent name.

In [ ]:
# Shared memory isa list of dicts. Each agent reads it and appends.
# This gets reset per session in run_session().

def format_memory(memory: list) -> str:
    """Format the shared memory list as a readable string for agent prompts."""
    if not memory:
        return ""
    parts = []
    for entry in memory:
        parts.append(f"[{entry['agent']}] {entry['role'].title()}: {entry['content']}")
    return "\n".join(parts)

## Step 3 — Instrumented Memory Hook

Reads from and writes to the shared Python list. Records latency and context for metrics.

In [ ]:
class ListMemoryHook(HookProvider):
    """Hook that reads/writes a shared Python list instead of AgentCore Memory."""

    def __init__(self, memory: list, collector: MetricsCollector, agent_name: str):
        self.memory = memory
        self.collector = collector
        self.agent_name = agent_name

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Read shared memory and inject into system prompt."""
        t0 = time.perf_counter()
        context = format_memory(self.memory)
        read_latency = time.perf_counter() - t0
        self.collector.record_memory_read_latency(self.agent_name, read_latency)

        if context:
            event.agent.system_prompt += (
                f"\n\nShared memory from other agents:\n{context}"
                "\n\nUse this context. Reference specific details from other agents.")
            logger.info(f"[{self.agent_name}] Read {len(self.memory)} entries from memory")

        self.collector.record_retrieved_context(self.agent_name, context)

    def on_message_added(self, event: MessageAddedEvent):
        """Write agent response to shared memory."""
        last = event.agent.messages[-1]

        # Record response for metrics
        if last["role"] == "assistant":
            self.collector.record_response(self.agent_name, last["content"][0]["text"])

        # Write to shared memory
        t0 = time.perf_counter()
        self.memory.append({
            "agent": self.agent_name,
            "role": last["role"],
            "content": last["content"][0]["text"],
            "ts": time.time(),
        })
        self.collector.record_memory_write_latency(self.agent_name, time.perf_counter() - t0)

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

## Step 4 — Session Runner

Three spokes + one hub coordinator. Each session gets a fresh shared memory list.

In [ ]:
def run_session(conversation: list, session_label: str) -> MetricsCollector:
    """Run a full conversation session and return the metrics collector."""
    shared_memory = []  # fresh memory per session
    collector = MetricsCollector(region=region)
    turn_counter = 0

    @tool
    def flight_booking_assistant(query: str) -> str:
        """Process flight booking queries.
        Args:
            query: A flight-related question
        Returns:
            Flight information and booking options
        """
        collector.record_handoff("flight", query)
        hook = ListMemoryHook(shared_memory, collector, "flight")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=FLIGHT_PROMPT)
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("flight", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("flight",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def hotel_booking_assistant(query: str) -> str:
        """Process hotel booking queries.
        Args:
            query: A hotel-related question
        Returns:
            Hotel information and booking options
        """
        collector.record_handoff("hotel", query)
        hook = ListMemoryHook(shared_memory, collector, "hotel")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=HOTEL_PROMPT)
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("hotel", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("hotel",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def itinerary_assistant(query: str) -> str:
        """Build a travel itinerary from flight and hotel results.
        Args:
            query: Request to build an itinerary
        Returns:
            A cohesive travel itinerary
        """
        collector.record_handoff("itinerary", query)
        hook = ListMemoryHook(shared_memory, collector, "itinerary")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=ITINERARY_PROMPT)
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("itinerary", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("itinerary",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    hub = Agent(system_prompt=HUB_PROMPT, model=AGENT_MODEL_ID,
               tools=[flight_booking_assistant, hotel_booking_assistant, itinerary_assistant])

    for msg in conversation:
        turn_counter += 1
        collector.begin_turn(turn_counter, msg)
        print(f"\n{'='*60}")
        print(f"[{session_label}] Turn {turn_counter}: {msg}")
        print(f"{'='*60}")
        resp = hub(msg)
        collector.end_turn()
        print(f"\nHub: {str(resp)[:300]}...")

    return collector, shared_memory

## Step 5 — Simple Session

Fixed budget, no mid-session changes. This is the baseline.

In [ ]:
simple_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
    "Now build me a day-by-day itinerary for the trip based on the flight and hotel you found.",
]

simple_metrics, simple_memory = run_session(simple_conversation, "simple")

### Simple Session — Context Flow Trace

Shows what each agent received (handoff + memory read) and produced (response). Follow the flow to see how context moves through the system.

In [ ]:
display(Markdown(simple_metrics.trace_report()))

### Simple Session — Shared Memory Contents

Raw contents of the shared memory list after the session completes.

In [ ]:
print(f"Memory has {len(simple_memory)} entries:\n")
for i, entry in enumerate(simple_memory):
    print(f"  [{i}] agent={entry['agent']}, role={entry['role']}")
    print(f"      {entry['content'][:200]}...\n")

## Step 6 — Session with Feedback

Same initial request, but the user changes the budget mid-session. This tests:
- Does the coordinator re-dispatch with the updated budget?
- Do the spokes read the latest context from memory?
- Does the itinerary agent produce a plan consistent with the new budget?

In [ ]:
feedback_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
    "Actually, my budget changed to $1400 total for flights. "
    "Please redo the flight and hotel search with this new budget.",
    "Now build me a day-by-day itinerary for the trip based on the updated flight and hotel.",
]

feedback_metrics, feedback_memory = run_session(feedback_conversation, "feedback")

### Feedback Session — Context Flow Trace

In [ ]:
display(Markdown(feedback_metrics.trace_report()))

### Feedback Session — Shared Memory Contents

In [ ]:
print(f"Memory has {len(feedback_memory)} entries:\n")
for i, entry in enumerate(feedback_memory):
    print(f"  [{i}] agent={entry['agent']}, role={entry['role']}")
    print(f"      {entry['content'][:200]}...\n")

## Step 7 — Run LLM-as-Judge Evaluations

Runs Claude Opus as judge on every agent call in both sessions. Evaluates context freshness, handoff completeness, utilization, write accuracy, redundancy, and cross-agent consistency.

In [ ]:
print("Evaluating simple session...")
simple_metrics.evaluate_all()

print("Evaluating feedback session...")
feedback_metrics.evaluate_all()

## Step 8 — Results: Simple Session

In [ ]:
display(Markdown("## Simple Session\n\n" + simple_metrics.context_metrics_report()))
display(Markdown(simple_metrics.latency_metrics_report()))

## Step 9 — Results: Session with Feedback

In [ ]:
display(Markdown("## Session with Feedback\n\n" + feedback_metrics.context_metrics_report()))
display(Markdown(feedback_metrics.latency_metrics_report()))

## Step 10 — Side-by-Side Comparison

In [ ]:
display(Markdown(MetricsCollector.comparison_report(
    simple_metrics, feedback_metrics,
    label_a="Simple Session", label_b="Session with Feedback")))

## Visual Analysis

In [ ]:
import matplotlib.pyplot as plt
from eval_helpers import (
    plot_context_metrics_radar, plot_latency_breakdown,
    plot_session_comparison
)

In [ ]:
fig = plot_context_metrics_radar(simple_metrics, "Simple Session")
plt.show()

fig = plot_context_metrics_radar(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_latency_breakdown(simple_metrics, "Simple Session")
plt.show()

fig = plot_latency_breakdown(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_session_comparison(simple_metrics, feedback_metrics,
                              "Simple Session", "Feedback Session")
plt.show()

In [ ]:
feedback_memory